# NAOMI encoder training arms (Colab)

Runs `scripts/run_encoder_arms.sh` -- the encoder scaling-replication arms from `dev/AUDIT_2026-09-08.md` finding 7 + recommendation (c):

- **v2_788**: `runs/encoder_gold_v2.jsonl`, n_train=788 (today's data, today's size)
- **v3_788**: `runs/encoder_gold_v3.jsonl`, n_train=788 (same size as v2_788, CLEANER data only)
- **v3_3000**: `runs/encoder_gold_v3.jsonl`, n_train=3000 (v3_788 + MORE data)

each x seeds {0, 1}, all trained for the SAME optimizer-step budget (`--max-steps`, default 40000) rather than a fixed epoch count, and all scored on the SAME held-out sentences (`runs/holdout_sentences.txt`, `scripts/make_holdout.py`) against both v2 and v3 targets. This isolates "more data" from "cleaner data" instead of bundling both into one uncontrolled retrain.

**`runs/encoder_gold_v3.jsonl` does not exist yet.** Until it is built and fetched, the v3_788 / v3_3000 arms are skipped automatically (clear message, not a crash) and only v2_788 runs. Run the v3 gold build first if you want the full 6-arm comparison.

This mirrors `colab/Encoder_Train.ipynb`'s setup cells (repo clone, deps, USVS build, gold-data fetch) -- run the cells below in order. **This is many CPU-hours at the default `--max-steps 40000`** (see the throughput estimate the runner prints before it starts training) -- the lead should confirm the step budget and which arms to run before letting the last cell go unattended.

In [ ]:
!git clone -b encoder-train-arms https://github.com/LinguisticsDevelopment/NAOMI.git && cd NAOMI/consciousness_transformer && pip install -q torch numpy nltk pytest && pip install -e .

In [ ]:
%cd NAOMI/consciousness_transformer
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
!python -c "import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); nltk.download('omw-2.0', quiet=True)"
!python scripts/build_usvs.py
!mkdir -p runs
!git show origin/encoder-gold-v2:consciousness_transformer/runs/encoder_gold_v2.jsonl > runs/encoder_gold_v2.jsonl
!wc -l runs/encoder_gold_v2.jsonl

In [ ]:
# Optional: fetch the v3 gold file once it exists on some branch (adjust the ref below),
# so the v3_788 / v3_3000 arms are runnable. If you skip this cell, run_encoder_arms.sh
# will still run v2_788 and print a clear skip message for the v3 arms.
V3_REF = None  # e.g. "origin/encoder-gold-v3"
if V3_REF:
    !git show {V3_REF}:consciousness_transformer/runs/encoder_gold_v3.jsonl > runs/encoder_gold_v3.jsonl
    !wc -l runs/encoder_gold_v3.jsonl
else:
    print("V3_REF not set -- runs/encoder_gold_v3.jsonl left absent; v3 arms will be skipped.")

In [ ]:
# Shared held-out sentences (item C): the CURRENT seeded v2 test split (n~98),
# the exact set the existing 0.70/0.68 numbers were measured on, plus its dev split.
!python scripts/make_holdout.py

In [ ]:
# Dry run first: prints the 6 planned commands (v3 arms marked [SKIP] if
# runs/encoder_gold_v3.jsonl is absent) without training anything.
!bash scripts/run_encoder_arms.sh --dry-run

In [ ]:
# The lead runs this cell. Override STEPS / PARALLEL as needed, e.g.:
#   !STEPS=40000 bash scripts/run_encoder_arms.sh --parallel 2
# Sequential by default; logs land in runs/arms/<arm>_<seed>.log, results in runs/arms/summary.tsv.
!bash scripts/run_encoder_arms.sh

In [ ]:
import pandas as pd
pd.read_csv("runs/arms/summary.tsv", sep="\t")

## Reading `runs/arms/summary.tsv`

One row per (arm, seed): `optimizer_steps`/`stop_reason` confirm every arm actually trained to the same step budget (not a fixed-epoch confound, dev/AUDIT_2026-09-08.md finding 7); `holdout_v2_*` and `holdout_v3_*` are `nsm_ct.encoder_train_util.evaluate_full`'s best-of-k edge precision/recall plus the audit's rank-1 committed-tree edge-F1 and mean forest width (finding 5 + recommendation (b)), scored on the identical held-out sentences regardless of which gold file trained the arm. Compare `v2_788` vs `v3_788` (data quality at equal size) and `v3_788` vs `v3_3000` (data quantity at equal quality); a checkpoint path is included in each row for a deeper `scripts/rescore_encoder.py` pass.